In [2]:
import pandas as pd
import duckdb
data = [
    # User A
    ["A", "2026-07-01 09:00:00", "O001", 100],
    ["A", "2026-07-01 10:30:00", "O002", 80],
    ["A", "2026-07-01 14:00:00", "O003", 120],
    ["A", "2026-07-02 09:20:00", "O004", 60],

    # User B
    ["B", "2026-07-01 09:20:00", "O005", 60],
    ["B", "2026-07-01 11:00:00", "O006", 90],
    ["B", "2026-07-02 16:00:00", "O007", 150],

    # User C
    ["C", "2026-07-01 10:00:00", "O008", 200],
    ["C", "2026-07-01 15:30:00", "O009", 50],
    ["C", "2026-07-02 12:00:00", "O010", 75],
]

df = pd.DataFrame(
    data,
    columns=["user_id", "order_time", "order_id", "amount"]
)

df["order_time"] = pd.to_datetime(df["order_time"])

print(df)



  user_id          order_time order_id  amount
0       A 2026-07-01 09:00:00     O001     100
1       A 2026-07-01 10:30:00     O002      80
2       A 2026-07-01 14:00:00     O003     120
3       A 2026-07-02 09:20:00     O004      60
4       B 2026-07-01 09:20:00     O005      60
5       B 2026-07-01 11:00:00     O006      90
6       B 2026-07-02 16:00:00     O007     150
7       C 2026-07-01 10:00:00     O008     200
8       C 2026-07-01 15:30:00     O009      50
9       C 2026-07-02 12:00:00     O010      75


## 题目要求

### 分别使用 SQL 和 Pandas 完成：

- 计算每个用户每次下单时，最近 2 次订单的平均消费金额。

**注意：**

- 最近 2 次 = 当前订单 + 上一笔订单。
- 如果当前用户还只有 1 笔订单，就只用当前这一笔计算平均值。

### 最终输出字段：

- `user_id`
- `order_time`
- `order_id`
- `amount`
- `moving_avg_2_amount`

In [ ]:
# SQL轨道 

query = """

SELECT
    user_id,
    order_time,
    order_id,
    amount,
    AVG(amount) OVER(
        PARTITION BY user_id
        ORDER BY order_time,order_id 
        ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
        ) AS moving_avg_2_amount
FROM df
order by user_id,order_time,order_id
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,user_id,order_time,order_id,amount,moving_avg_2_amount
0,A,2026-07-01 09:00:00,O001,100,100.0
1,A,2026-07-01 10:30:00,O002,80,90.0
2,A,2026-07-01 14:00:00,O003,120,100.0
3,A,2026-07-02 09:20:00,O004,60,90.0
4,B,2026-07-01 09:20:00,O005,60,60.0
5,B,2026-07-01 11:00:00,O006,90,75.0
6,B,2026-07-02 16:00:00,O007,150,120.0
7,C,2026-07-01 10:00:00,O008,200,200.0
8,C,2026-07-01 15:30:00,O009,50,125.0
9,C,2026-07-02 12:00:00,O010,75,62.5


In [9]:
# PANDAS轨道

df_pd = (
    df
    .sort_values(by=['user_id','order_time','order_id'])
    .assign(
        moving_avg_2_amount = lambda x:(
            x.groupby('user_id')['amount']
            .rolling(2,min_periods=1)
            .mean()
            .reset_index(level=0,drop=True)
        )
    )
)
df_pd

,user_id,order_time,order_id,amount,moving_avg_2_amount
0,A,2026-07-01 09:00:00,O001,100,100.0
1,A,2026-07-01 10:30:00,O002,80,90.0
2,A,2026-07-01 14:00:00,O003,120,100.0
3,A,2026-07-02 09:20:00,O004,60,90.0
4,B,2026-07-01 09:20:00,O005,60,60.0
5,B,2026-07-01 11:00:00,O006,90,75.0
6,B,2026-07-02 16:00:00,O007,150,120.0
7,C,2026-07-01 10:00:00,O008,200,200.0
8,C,2026-07-01 15:30:00,O009,50,125.0
9,C,2026-07-02 12:00:00,O010,75,62.5
